# Imports

In [ ]:
import os
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords as nltk_stopwords
import re
import json
import spacy
import json
import pandas as pd
from collections import Counter
from nltk.util import ngrams
import unicodedata
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
import ast
import xlsxwriter

# Preprocessing

In [ ]:
with open('stopwords-iso.json', 'r', encoding='utf-8') as file:
    stopwords_iso = json.load(file)

stopwords = set(nltk_stopwords.words('english'))
custom_stopwords = set(['server', 'joined', 'scroll', 'papyrus', 'image', 'brett', 'olsen', 'entry', 'start', 'thread', 'moshe', 'levy', 'casey', 'handmer', 'mae', 'sawatzky','hari','seldon','ben'])
stopwords.update(custom_stopwords)
stopwords.update(stopwords_iso['en'])

def preprocess_text(text):

    # Convert text to lowercase
    text = text.lower()
    
    # Load SpaCy model
    nlp = spacy.load("en_core_web_sm")
    doc = nlp(text)

    # Lemmatize and lowercase each token
    text = ' '.join([token.lemma_.lower() for token in doc])
    
    # Remove URLs including bare domains and www-prefixed links
    text = re.sub(r'\b(?:https?://|www\.)\S+\b', '', text)
    text = re.sub(r'\b\S+\.(com|org|net|edu|gov|io|co|us|uk)\b', '', text)

    # Remove file names with specific extensions
    text = re.sub(r'\b\w+\.(zip|tif|tiff|pdf|jpg|jpeg|png|gif|docx|xlsx|rar|txt|csv|json|obj)\b[^\w\s]*', '', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)

    # Remove special characters and emojis - could also be causing issues with stopwords, need to look into it
    text = re.sub(r'[\U0001F600-\U0001F64F'
              r'\U0001F300-\U0001F5FF'  
              r'\U0001F680-\U0001F6FF'  
              r'\U0001F700-\U0001F77F'  
              r'\U0001F780-\U0001F7FF'  
              r'\U0001F800-\U0001F8FF'  
              r'\U0001F900-\U0001F9FF'  
              r'\U0001FA00-\U0001FA6F'  
              r'\U0001FA70-\U0001FAFF'  
              r'\u2600-\u26FF'          
              r'\u2700-\u27BF'       
              ']+', '', text)

    # Remove stopwords
    tokens = text.split()
    tokens = [token for token in tokens if token not in stopwords]
    
    return ' '.join(tokens)

def preprocess_json(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as file:
        data = json.load(file)

    for message in data.get('messages', []):
        if message.get('content'):
            message['content'] = preprocess_text(message['content'])

    with open(output_file, 'w', encoding='utf-8') as file:
        json.dump(data, file, indent=4)

    print(f'Preprocessed data has been saved to {output_file}')

 
def preprocess_all_files(input_folder, output_folder):
    for filename in os.listdir(input_folder):
        input_file = os.path.join(input_folder, filename)
        output_file = os.path.join(output_folder, filename)
        preprocess_json(input_file, output_file)


input_folder = 'filteredJSON'
output_folder = 'preprocessedJSON'

preprocess_all_files(input_folder, output_folder)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\summerm\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Preprocessed data has been saved to preprocessedJSON\Vesuvius Challenge - code - 3D FFT [1165255454037917797]_filtered.json
Preprocessed data has been saved to preprocessedJSON\Vesuvius Challenge - code - Code tip_ mirror the data server directory structure locally [1162854815731306666]_filtered.json
Preprocessed data has been saved to preprocessedJSON\Vesuvius Challenge - code - Colab recommendation for persistent storage_ [1163820413621633124]_filtered.json
Preprocessed data has been saved to preprocessedJSON\Vesuvius Challenge - code - Constraint solvers for bruteforcing missing_ambiguous letters [1163049828721369118]_filtered.json
Preprocessed data has been saved to preprocessedJSON\Vesuvius Challenge - code - Permission denied when sshing from paperspace machine (web browser works okay) [1163661867185606737]_filtered.json
Preprocessed data has been saved to preprocessedJSON\Vesuvius Challenge - code - Speed-check your image loading [1164605201408348251]_filtered.json
Preprocessed 

# Organizing Messages

In [ ]:
input_folder = 'preprocessedJSON'
original_folder = 'filteredJSON'

def get_all_documents(preprocessed_folder, original_folder):
    all_docs = []

    for fname in os.listdir(preprocessed_folder):
        # if fname not in files:
        #     continue

        # Load original messages from filteredJSON
        with open(os.path.join(original_folder, fname), 'r', encoding='utf-8') as f_orig:
            original_data = json.load(f_orig)
            original_msgs = {
                msg["timestamp"]: msg.get("content", "")
                for msg in original_data.get("messages", [])
            }

        # Load preprocessed messages from preprocessedJSON
        with open(os.path.join(preprocessed_folder, fname), 'r', encoding='utf-8') as f_proc:
            data = json.load(f_proc)
            channel = data.get("channel", "Unknown")
            channel_name = channel.get("name") if isinstance(channel, dict) else channel

            for msg in data.get("messages", []):
                timestamp = msg.get("timestamp", "Unknown")
                content = msg.get("content", "")
                raw_content = original_msgs.get(timestamp, "")  # Match by timestamp

                if content.strip():  # Only include non-empty messages
                    all_docs.append({
                        "channel": channel_name,
                        "user": msg.get("author", {}).get("name", "Unknown"),
                        "timestamp": timestamp,
                        "content": content,
                        "raw_content": raw_content
                    })

    return all_docs

df = pd.DataFrame(get_all_documents(input_folder, original_folder))
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
if df['timestamp'].dt.tz is not None:
    df['timestamp'] = df['timestamp'].dt.tz_localize(None)


# Term Frequency

In [ ]:
# Load SpaCy model
spacy.cli.download("en_core_web_sm")
nlp = spacy.load("en_core_web_sm")


def is_greek_script(token_text):
    return any('GREEK' in unicodedata.name(c, '') for c in token_text)


def tokenize(text):
    doc = nlp(text.lower())
    tokens = []

    for token in doc:
        if token.is_alpha:
            if is_greek_script(token.text):
                tokens.append('[GREEK_LANGUAGE]')
            else:
                tokens.append(token.text)
    return tokens


df['tokens'] = df['content'].apply(tokenize)

# Compute term frequency and document frequency
def get_ngram_stats(token_lists, n=1):
    term_freq = Counter()
    doc_freq = Counter()

    for tokens in token_lists:
        if n == 1:
            terms = tokens
        else:
            terms = list(ngrams(tokens, n))

        term_freq.update(terms)
        doc_freq.update(set(terms))  # Count each term once per document

    return term_freq, doc_freq


# Convert stats to DataFrame
def ngram_df_with_stats(term_freq, doc_freq, label='ngram', top_k=50):
    df = pd.DataFrame({
        label: list(term_freq.keys()),
        'term_frequency': list(term_freq.values()),
        'document_frequency': [doc_freq[term] for term in term_freq]
    })
    df.sort_values(by='term_frequency', ascending=False, inplace=True)
    return df.head(top_k)


# Combine all messages from all channels
all_msgs = df['tokens'].dropna().tolist()

# Get stats for all combined messages
tf_uni_all, df_uni_all = get_ngram_stats(all_msgs, n=1)
tf_bi_all, df_bi_all = get_ngram_stats(all_msgs, n=2)

# Write to Excel
with pd.ExcelWriter('channel_analysis_all_combined.xlsx') as writer:
    # Original messages sheet
    df.to_excel(writer, sheet_name='Messages', index=False)

    # Combined unigrams
    ngram_df_with_stats(tf_uni_all, df_uni_all, 'unigram').to_excel(writer, sheet_name='Unigrams', index=False)

    # Combined bigrams
    ngram_df_with_stats(tf_bi_all, df_bi_all, 'bigram').to_excel(writer, sheet_name='Bigrams', index=False)

print("Channel analysis with all channels combined has been saved to 'channel_analysis_all_combined.xlsx'")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Channel analysis with all channels combined has been saved to 'channel_analysis_all_combined.xlsx'


# Statistics

In [ ]:
# Read in the dataframe
df = pd.read_excel('channel_analysis.xlsx', sheet_name='Messages')

# Normalize channel names
df['channel'] = df['channel'].str.lower()

# Define subsets
channels = {
    'General': df[df['channel'] == 'general'],
    'Papyrology': df[df['channel'] == 'papyrology'],
    'Both': df
}

# Loop through each subset
for name, subset in channels.items():
    print(f"\n--- {name} Channel ---")
    print(f"Total messages: {len(subset)}")

    # Convert timestamp and sort
    subset = subset.copy()
    subset['timestamp'] = pd.to_datetime(subset['timestamp'], errors='coerce')
    subset = subset.dropna(subset=['timestamp'])
    subset = subset.set_index('timestamp').sort_index()

    # Messages per day
    messages_per_day = subset.resample('D').size()

    # Reindex to include all days in range (even with zero messages)
    start_date = subset.index.min().normalize()
    end_date = subset.index.max().normalize()
    all_days = pd.date_range(start=start_date, end=end_date, freq='D')
    messages_per_day = messages_per_day.reindex(all_days, fill_value=0)

    # Plot with narrow bars and month labels
    plt.figure(figsize=(18, 6))
    plt.bar(messages_per_day.index, messages_per_day.values, width=0.8, color='skyblue')
    plt.title(f'{name} Messages Over Time')
    plt.xlabel('Month')
    plt.ylabel('Number of Messages')

    # Format x-axis to show month names
    ax = plt.gca()
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

    plt.tight_layout()
    plt.show()

    # Activity by weekday and hour
    subset['weekday'] = subset.index.weekday
    subset['hour'] = subset.index.hour
    activity = subset.groupby(['weekday', 'hour']).size().unstack(fill_value=0)

    plt.figure(figsize=(14, 6))
    sns.heatmap(activity, cmap='YlGnBu')
    plt.title(f'{name} Activity by Weekday and Hour')
    plt.xlabel('Hour of Day')
    plt.ylabel('Weekday (0=Monday)')
    plt.tight_layout()
    plt.show()

    # Top users
    top_users = subset['user'].value_counts().head(10)
    plt.figure(figsize=(14, 6))
    sns.barplot(x=top_users.values, y=top_users.index, palette='viridis')
    plt.title(f'{name} Top 10 Users by Number of Messages')
    plt.xlabel('Number of Messages')
    plt.ylabel('User')
    plt.tight_layout()
    plt.show()

# Stopword Removal Differences

In [ ]:
# SUMMERS output
df_revised = pd.read_excel("channel_analysis_all_combined.xlsx", sheet_name='Messages')

# NIKHILS output
df_streamlined = pd.read_excel("term_level_analysis5.xlsx", sheet_name="messages_cleaned")

# Build alignment key from raw message text
def build_key(row):
    raw = str(row.get("raw_content", row.get("raw", ""))).strip().lower()
    return raw[:100]

df_revised["msg_key"] = df_revised.apply(build_key, axis=1)
df_streamlined["msg_key"] = df_streamlined.apply(build_key, axis=1)

# Merge on msg_key and include full messages
df_merged = pd.merge(
    df_revised[["msg_key", "tokens", "raw_content"]],
    df_streamlined[["msg_key", "context", "raw"]],
    on="msg_key",
    how="inner"
)

# Parse token lists (preserving as lists, not sets)
def parse_tokens_list(val):
    try:
        parsed = ast.literal_eval(val)
        # Convert tuples (bigrams) to strings for comparison
        if parsed and isinstance(parsed[0], tuple):
            return [' '.join(map(str, item)) for item in parsed]
        return parsed
    except:
        return []

def parse_context_list(val):
    return [t.strip() for t in str(val).split(",") if t.strip()]

df_merged["tokens_list"] = df_merged["tokens"].apply(parse_tokens_list)
df_merged["context_list"] = df_merged["context"].apply(parse_context_list)

# Compare lists (counting duplicates and order)
def compare_lists(row):
    tokens_list = row["tokens_list"]
    context_list = row["context_list"]
    
    # Create frequency counters
    from collections import Counter
    tokens_counter = Counter(tokens_list)
    context_counter = Counter(context_list)
    
    # Find differences
    only_in_revised = []
    only_in_streamlined = []
    
    # Tokens only in revised
    for token, count in tokens_counter.items():
        context_count = context_counter.get(token, 0)
        if count > context_count:
            only_in_revised.extend([token] * (count - context_count))
    
    # Tokens only in streamlined
    for token, count in context_counter.items():
        tokens_count = tokens_counter.get(token, 0)
        if count > tokens_count:
            only_in_streamlined.extend([token] * (count - tokens_count))
    
    return pd.Series({
        "only_in_revised": str(sorted(only_in_revised)),
        "only_in_streamlined": str(sorted(only_in_streamlined)),
        "diff_count": len(only_in_revised) + len(only_in_streamlined)
    })

df_merged = pd.concat([df_merged, df_merged.apply(compare_lists, axis=1)], axis=1)

# Export all messages with diff_count
with pd.ExcelWriter("token_comparison_all_with_diff.xlsx", engine='xlsxwriter') as writer:
    df_merged.to_excel(writer, index=False, sheet_name="All Messages")
    workbook = writer.book
    worksheet = writer.sheets["All Messages"]
    
    wrap_format = workbook.add_format({'text_wrap': True})
    worksheet.set_column("A:H", 60, wrap_format)

# Export only mismatches
mismatches = df_merged[df_merged["diff_count"] > 0]
with pd.ExcelWriter("token_comparison_mismatches_only.xlsx", engine='xlsxwriter') as writer:
    mismatches.to_excel(writer, index=False, sheet_name="Mismatches")
    workbook = writer.book
    worksheet = writer.sheets["Mismatches"]
    
    wrap_format = workbook.add_format({'text_wrap': True})
    worksheet.set_column("A:H", 60, wrap_format)

print(f"Total messages compared: {len(df_merged)}")
print(f"Messages with differences: {len(mismatches)}")

Total messages compared: 16008
Messages with differences: 769
